## 1. Defining cleaning criteria
The exploration notebook identified several data-quality issues that need to be addressed before the analysis.  
The cleaning criteria are:  
- Target vessel types: retain Cargo, Tanker, and Passenger vessels only  
- Valid vessel identity: investigate and remove malformed or non-vessel MMSI records  
- Present and realistic SOG values: remove missing speeds and investigate physically impossible speeds  
- Relevant navigational status: exclude observations where vessels are stationary or otherwise unsuitable for measuring voyage progress, such as vessels at anchor or moored  
- Valid destination: exclude unsuitable destinations such as 'Unknown' and 'FOR ORDERS'  
- Destination normalization: standardize different names and codes that refer to the same port  
- ETA consistency: ensure reported ETA values are parseable and logically valid relative to each observation timestamp and voyage information  

## Target Vessel Types
We simply keep Cargo, Tanker and passenger vessels

In [ ]:
import pandas as pd

cols = ['MMSI', 'Latitude', 'Longitude', 'SOG', 'Ship type', 'Destination', 'ETA', '# Timestamp', 'Type of mobile', 'Navigational status', 'Name']

df_ais = pd.read_csv('../data/raw/aisdk-2024-08-07.csv', usecols=cols)

df_target = df_ais[df_ais['Ship type'].isin(['Cargo', 'Tanker', 'Passenger'])]
rows_before = df_ais.shape[0]
rows_target = df_target.shape[0]

print(rows_before, rows_target)

We can see that just by filtering out vessel types we are not interested in we get back 11,842,463 observations/rows of data from the original 28,203,137

## Present And Realistic SOG Values
Missing and physically implausible speeds are further investigated and removed

In [ ]:
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
print(df_target['SOG'].describe())

As we saw in the exploration notebook there is a max value of 99.7 knots. Lets dig deeper to investigate the vessel involved


In [ ]:
speed_anomalies = df_target[df_target['SOG'] > 40]
print(speed_anomalies.shape[0])
print(speed_anomalies['MMSI'].nunique())
print(speed_anomalies.groupby('MMSI')['SOG'].max())
print(speed_anomalies[speed_anomalies['MMSI'] == 229191000])
print(speed_anomalies[speed_anomalies['MMSI'] == 352179000 ])

These two vessels involved contain inaccurate all day records with one being labelled as 'At Anchor' while having SOG values over 50 knots and the other one having physically implausible SOG values. Based on these findings we'll remove them and continue investigating for any other suspicious speeds that could potentially affect data quality.

In [ ]:
df_target = df_target[(df_target['MMSI'] != 229191000) & (df_target['MMSI'] != 352179000)]
speed_anomalies = df_target[(df_target['SOG'] > 30) & (df_target['SOG'] < 40)]
print(speed_anomalies.shape[0])
print(speed_anomalies['MMSI'].nunique())
print(speed_anomalies.groupby('MMSI')['SOG'].max())
print(speed_anomalies[speed_anomalies['MMSI'] == 230712000])
print(speed_anomalies[speed_anomalies['MMSI'] == 257182700])

Upon further inspection of vessels with SOG values within the 30-40kn range we've found two different cases. A cargo vessel with reported SOG values around 30-32kn while its coordinates remained unchanged, indicating unreliable movement. The second one is a passenger vessel with reported SOG values between 30 and 36kn and normal position movement. Despite its movement appearing plausible it lacks ETA data deeming it unfit for our analysis. In conclusion both vessels found in the 30-40kn range will be excluded.

In [5]:
df_target = df_target[(df_target['MMSI'] != 230712000) & (df_target['MMSI'] != 257182700)]

## Relevant Navigational Status
Observations where vessels are stationary/unsuitable for measuring voyage progress will be removed. 
!important distinction: we only exlcude individual observations and not whole MMSIs because a vessel can be stationary for a part of its voyage and moving for another therefore deeming it suitable for voyage progress tracking

In [ ]:
print(df_target['Navigational status'].value_counts())
stationary_stat = ['Under way using engine', 'Constrained by her draught', 'Under way sailing', 'Restricted maneuverability']
df_target = df_target[df_target['Navigational status'].isin(stationary_stat)]

## Valid Vessel Identity
Observations with malformed or non-vessel MMSIs will be investigated and removed. Standard class A vessel MMSIs have 9 digits

In [ ]:
df_target['MMSI'] = df_target['MMSI'].astype(str)
print(df_target['MMSI'].str.len().value_counts())

Our previous finding from the data exploration notebook that some MMSIs were malformed and non-standard turned out to be false. The vessel type filtering we did above removed all the weird base station non-vessel MMSIs. Another useful check would be to see if each MMSI corresponds to a single vessel

In [ ]:
df_target.groupby('MMSI')['Name'].nunique().sort_values(ascending=False)
print(df_target['Name'][df_target['MMSI'] == '218795000'])
print(df_target.loc[df_target['MMSI'] == '218795000', 'Name'].unique())
print(df_target['Name'][df_target['MMSI'] == '212499000'])
print(df_target.loc[df_target['MMSI'] == '212499000', 'Name'].unique())

Two MMSIs are associated with two slightly different vessel names. Further investigation showed these are corrupted variants of the same vessel names rather than distinct vessels [ROBIN HOOD / ROBIN HOOI6T] and [NILS DACKE / NILS DACKI)K]. These two MMSIs will be therefore included in the analysis

## Valid Destination
Observations with irrelevant/non-specific destination values like 'Unknown' and 'For Orders' will be removed

In [ ]:
print(df_target['Destination'].value_counts().head(50))
invalid_dests = ['Unknown', 'FOR ORDERS', 'FOR ORDER']
df_target = df_target[~df_target['Destination'].isin(invalid_dests)]

After investigating the top 50 most used destination values we have removed observations containining values such as 'Unknown' , 'FOR ORDERS' , 'FOR ORDER'. We also found a specific destination format worth mentioning: SEMMA<>DETRV, PLGDND>DEBRV. These seem to be route-like strings(DEPARTED FROM > DESTINATION).

## Destination Normalization
Standardize different names and codes that refer to the same port  

In [ ]:
print(df_target['Destination'].value_counts().head(50))
before = df_target['Destination'].nunique()
print(before)

Many destinations have whitespaces splitting the port code in half so 'PLGDY' and 'PL GDY' are the exact same destination. In order to make our normalization attempt a bit more structured and easy we'll go ahead and remove all whitespaces aswell as any lowercase characters.

In [ ]:
df_target['Destination'] = df_target['Destination'].str.upper().str.strip().str.replace(' ', '')
print(df_target['Destination'].value_counts().head(50))
after = df_target['Destination'].nunique()
print(after)

We have succesfully made the number of destinations drop by 56(from 772 down to 716). Next we'll inspect the new top normalized destination values and decide which ones are worth merging.

In [ ]:
dest_count = df_target['Destination'].value_counts()
print(f"{((dest_count.head(50).sum() / dest_count.sum()) * 100):.2f}%")
print(f"{((dest_count.head(100).sum() / dest_count.sum()) * 100):.2f}%")
print(f"{((dest_count.head(200).sum() / dest_count.sum()) * 100):.2f}%")

Out of the 716 total destination values 73.60% is represented by the top 200 destinations. Manually tuning every single variation of each destination would be a waste of time right now. We'll handle the remaining values later during port mapping.

## ETA Consistency
We'll ensure reported ETA values are parseable and logically valid relative to each observations timestamp and voyage information.

In [ ]:
df_target['# Timestamp'] = pd.to_datetime(df_target['# Timestamp'], errors='coerce')
df_target['ETA'] = pd.to_datetime(df_target['ETA'], errors='coerce')
print(df_target['# Timestamp'].isna().sum())
print(df_target['ETA'].isna().sum())
invalid_eta = df_target[df_target['ETA'] < df_target['# Timestamp']]
print(invalid_eta['ETA'].count())
invalid_eta['eta_diff'] = (invalid_eta['# Timestamp'] - invalid_eta['ETA'])
print(invalid_eta['eta_diff'].describe())

After parsing each ETA to datetime we see that a staggering 2,390,583 observations have unparseable ETA values. Furthermore another 688,064 observations have reported ETAs before their observation timestamp with an average time delta of 66 days. We'll use this criteria to filter out unsuitable observations.

In [ ]:
df_target = df_target[df_target['ETA'].notna()]
df_target = df_target[df_target['ETA'] > df_target['# Timestamp']]
print(df_target.shape[0])

After following our cleaning criteria we've gone from the original 28,203,137 observations down to 7,727,047. 

In [35]:
df_target.to_csv('../outputs/ais_cleaned.csv')